In [1]:
import pandas as pd

In [2]:
# importing the dataset
df = pd.read_csv('US_Accidents_March23.csv')
df.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


In [3]:
# List of columns to drop
columns_to_drop = [
    'ID', 'Source', 'Description','State',
    'Street', 'City', 'County', 'Zipcode', 'Country', 
    'Airport_Code', 'End_Lat', 'End_Lng', 'Weather_Timestamp','Civil_Twilight','Nautical_Twilight','Astronomical_Twilight'
]

# Drop columns
df = df.drop(columns=columns_to_drop)

# Check the shape of the cleaned dataset
print("Shape after dropping columns:", df.shape)

Shape after dropping columns: (7728394, 30)


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7728394 entries, 0 to 7728393
Data columns (total 30 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Severity           int64  
 1   Start_Time         object 
 2   End_Time           object 
 3   Start_Lat          float64
 4   Start_Lng          float64
 5   Distance(mi)       float64
 6   Timezone           object 
 7   Temperature(F)     float64
 8   Wind_Chill(F)      float64
 9   Humidity(%)        float64
 10  Pressure(in)       float64
 11  Visibility(mi)     float64
 12  Wind_Direction     object 
 13  Wind_Speed(mph)    float64
 14  Precipitation(in)  float64
 15  Weather_Condition  object 
 16  Amenity            bool   
 17  Bump               bool   
 18  Crossing           bool   
 19  Give_Way           bool   
 20  Junction           bool   
 21  No_Exit            bool   
 22  Railway            bool   
 23  Roundabout         bool   
 24  Station            bool   
 25  Stop              

In [5]:
# Check the missing entries in columns
missing_summary = df.isnull().sum()
print(missing_summary[missing_summary > 0])

Timezone                7808
Temperature(F)        163853
Wind_Chill(F)        1999019
Humidity(%)           174144
Pressure(in)          140679
Visibility(mi)        177098
Wind_Direction        175206
Wind_Speed(mph)       571233
Precipitation(in)    2203586
Weather_Condition     173459
Sunrise_Sunset         23246
dtype: int64


In [6]:
categorical_cols = ['Timezone', 'Wind_Direction', 'Weather_Condition', 'Sunrise_Sunset']
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

In [7]:
numerical_cols = ['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 
                  'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)  # Using median for the missing entries

In [8]:
df['Start_Time'] = pd.to_datetime(df['Start_Time'])
df['End_Time'] = pd.to_datetime(df['End_Time'])
df['duration_accident'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60
df['Start_Hour'] = df['Start_Time'].dt.hour
df['Start_Day'] = df['Start_Time'].dt.dayofweek
df['Start_Month'] = df['Start_Time'].dt.month
df = df.drop(columns=['End_Time','Start_Time'])
# Introduced the new feature 'duration_accident' which may help to find the the severity.

In [9]:
def optimize_memory(df):
    # Convert object columns to category
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].nunique() < 50:  # Threshold for categorical conversion
            df[col] = df[col].astype('category')
    
    # Convert bool columns to integers
    for col in df.select_dtypes(include=['bool']).columns:
        df[col] = df[col].astype('int8')
    
    # Downcast float and int columns
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    
    return df

df = optimize_memory(df)
print(df.info())  # Check reduced memory usage


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7728394 entries, 0 to 7728393
Data columns (total 32 columns):
 #   Column             Dtype   
---  ------             -----   
 0   Severity           int8    
 1   Start_Lat          float32 
 2   Start_Lng          float32 
 3   Distance(mi)       float32 
 4   Timezone           category
 5   Temperature(F)     float32 
 6   Wind_Chill(F)      float32 
 7   Humidity(%)        float32 
 8   Pressure(in)       float32 
 9   Visibility(mi)     float32 
 10  Wind_Direction     category
 11  Wind_Speed(mph)    float32 
 12  Precipitation(in)  float32 
 13  Weather_Condition  object  
 14  Amenity            int8    
 15  Bump               int8    
 16  Crossing           int8    
 17  Give_Way           int8    
 18  Junction           int8    
 19  No_Exit            int8    
 20  Railway            int8    
 21  Roundabout         int8    
 22  Station            int8    
 23  Stop               int8    
 24  Traffic_Calming    int8 

In [10]:
df.head()

,Severity,Start_Lat,Start_Lng,Distance(mi),Timezone,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),...,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,duration_accident,Start_Hour,Start_Day,Start_Month
0,3,39.865147,-84.058723,0.01,US/Eastern,36.900002,62.000000,91.0,29.680000,10.0,...,0,0,0,0,0,Night,314.0,5,0,2
1,2,39.928059,-82.831184,0.01,US/Eastern,37.900002,62.000000,100.0,29.650000,10.0,...,0,0,0,0,0,Night,30.0,6,0,2
2,2,39.063148,-84.032608,0.01,US/Eastern,36.000000,33.299999,100.0,29.670000,10.0,...,0,0,0,1,0,Night,30.0,6,0,2
3,3,39.747753,-84.205582,0.01,US/Eastern,35.099998,31.000000,96.0,29.639999,9.0,...,0,0,0,0,0,Night,30.0,7,0,2
4,2,39.627781,-84.188354,0.01,US/Eastern,36.000000,33.299999,89.0,29.650000,6.0,...,0,0,0,1,0,Day,30.0,7,0,2


In [11]:
# delete the rows where 30% of the features are unknown
threshold = 0.3
df = df.loc[:, df.isnull().mean() < threshold]

In [14]:
correlation_matrix = df.corr()

# Initialize a set to collect correlated feature pairs
correlated_features = set()

# Set the threshold for correlation
threshold = 0.8

# Iterate over the correlation matrix
for column in correlation_matrix:
    for index in correlation_matrix.index:
        if column != index and correlation_matrix[column][index] > threshold:  # Avoiding self-correlation
            correlated_features.add((column, index, correlation_matrix[column][index]))

# Print the correlated features
print("Correlated Features (threshold = {}):".format(threshold))
for feature1, feature2, corr in correlated_features:
    print(f"{feature1} and {feature2}: Correlation = {corr:.2f}")

Correlated Features (threshold = 0.8):
Wind_Chill(F) and Temperature(F): Correlation = 0.91
Temperature(F) and Wind_Chill(F): Correlation = 0.91


In [16]:
df.drop(columns=['Wind_Chill(F)'], inplace=True)

In [17]:
# Remove outliers
numerical_columns = df.select_dtypes(include=['float', 'int']).columns

for col in numerical_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

In [18]:
print(df.info())
print(df.head())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4430395 entries, 1 to 7728392
Data columns (total 31 columns):
 #   Column             Dtype   
---  ------             -----   
 0   Severity           int8    
 1   Start_Lat          float32 
 2   Start_Lng          float32 
 3   Distance(mi)       float32 
 4   Timezone           category
 5   Temperature(F)     float32 
 6   Humidity(%)        float32 
 7   Pressure(in)       float32 
 8   Visibility(mi)     float32 
 9   Wind_Direction     category
 10  Wind_Speed(mph)    float32 
 11  Precipitation(in)  float32 
 12  Weather_Condition  object  
 13  Amenity            int8    
 14  Bump               int8    
 15  Crossing           int8    
 16  Give_Way           int8    
 17  Junction           int8    
 18  No_Exit            int8    
 19  Railway            int8    
 20  Roundabout         int8    
 21  Station            int8    
 22  Stop               int8    
 23  Traffic_Calming    int8    
 24  Traffic_Signal     int8 

In [19]:
file_path = "us_road_accidents.csv"

# Save the dataset as a CSV file
df.to_csv(file_path, index=False)

print(f"Dataset saved to {file_path}")

Dataset saved to us_road_accidents.csv
